# Connect to Google Account

In [136]:
from google.colab import drive
drive.mount('/content/data', force_remount=True)

Mounted at /content/data


# import

In [137]:
!pip install -q catboost lightgbm xgboost

In [138]:
import os
import json
import numpy as np
import pandas as pd
import random
import torch

# 데이터 시각화에 사용할 라이브러리
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve
)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

#from stacking import StackingClassifier

In [139]:
# 브라우저에서 바로 그려지도록
%matplotlib inline

# 유니코드에서  음수 부호설정
mpl.rc('axes', unicode_minus=False)

# Global Variables

In [140]:
def reset_seeds(seed=42):
  random.seed(seed)
  os.environ['PYTHONHASHSEED'] = str(seed)      # 파이썬 환경변수 시드 고정
  np.random.seed(seed)
  torch.manual_seed(seed)                       # cpu 연산 무작위 고정
  torch.cuda.manual_seed(seed)                  # gpu 연산 무작위 고정
  torch.backends.cudnn.deterministic = True     # cuda 라이브러리에서 Deterministic(결정론적)으로 예측하기 (예측에 대한 불확실성 제거 )


In [141]:
import easydict
args = easydict.EasyDict()

reset_seeds() # 랜덤 고정!!

# path 정보
args.default_path = '/content/data/MyDrive/kaggle/data/'
args.train_csv = args.default_path+'train.csv'
args.test_csv = args.default_path+'test.csv'
args.default_submission_csv = args.default_path+'submission.csv'

# 제출할 파일 이름 양식 = _날짜_순
args.submission_path = '/content/data/MyDrive/kaggle/submission/'
args.submission_csv = 'submission.csv'
args.save_results = "model_results.json"

# 데이터 분석을 위한 변수들
args.random_state = 21
args.results = []

# Load Titanic


- Surived:0=사망, 1=생존
- Pclass: 1=1등석, 2=2등석, 3=3등석
- gender:male=남성, female=여성
- Age: 나이
- SibSp: 타이타닉 호에 동승한 자매/배우자의 수
- Parch: 타이타닉 호에 동승한 부모/자식의 수
- Ticket: 티켓 번호
- Fare: 승객 요금
- Cabin: 방 호수
- Embarked: 탑승지; C=셰르부르, Q=퀴즈타운, S=사우샘프턴

In [142]:
plt.style.use('fivethirtyeight')
plt.ion()

import warnings
warnings.filterwarnings('ignore')

In [143]:
ori_train = pd.read_csv(args.train_csv)
ori_test = pd.read_csv(args.test_csv)

# traindataset => feature+target
# test dataset => features + target은 없음 (예측해야함)
ori_train.shape, ori_test.shape

((916, 12), (393, 11))

In [144]:
pd.read_csv(args.default_submission_csv).shape

(393, 2)

In [145]:
ori_train.columns

Index(['passengerid', 'survived', 'pclass', 'name', 'gender', 'age', 'sibsp',
       'parch', 'ticket', 'fare', 'cabin', 'embarked'],
      dtype='object')

In [146]:
ori_train['passengerid'].nunique(), ori_train.shape[0]

(916, 916)

In [147]:
ori_train.drop('passengerid', axis=1, inplace=True)
ori_train.head()

,survived,pclass,name,gender,age,sibsp,parch,ticket,fare,cabin,embarked
0,0,2,"Wheeler, Mr. Edwin Frederick""""",male,NaN,0,0,SC/PARIS 2159,12.8750,NaN,S
1,0,3,"Henry, Miss. Delia",female,NaN,0,0,382649,7.7500,NaN,Q
2,1,1,"Hays, Mrs. Charles Melville (Clara Jennings Gr...",female,52.0,1,1,12749,93.5000,B69,S
3,1,3,"Andersson, Mr. August Edvard (""Wennerstrom"")",male,27.0,0,0,350043,7.7958,NaN,S
4,0,2,"Hold, Mr. Stephen",male,44.0,1,0,26707,26.0000,NaN,S


In [148]:
ori_test.set_index(['passengerid'], inplace=True)
print(f'{ori_test.shape}')

(393, 10)


# train_test_split

In [149]:
new_survived = pd.Categorical(ori_train["survived"])
new_survived = new_survived.rename_categories(["Died","Survived"])

new_survived.describe()

,counts,freqs
categories,,
Died,570,0.622271
Survived,346,0.377729


In [150]:
from sklearn.model_selection import train_test_split

In [151]:
y = ori_train['survived']
X = ori_train.drop(['survived'], axis=1)

In [152]:
reset_seeds()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=ori_train['survived'])
# 테스트 사이즈 축소

X_tr.shape, X_te.shape, y_tr.shape, y_te.shape

((641, 10), (275, 10), (641,), (275,))

# Data Preprocessing

In [153]:
train = X_tr.copy()       # 학습용 데이터셋 (features+target)
test = X_te.copy()        # 평가용 데이터셋 (features+target)
ori_te = ori_test.copy()  # 제출용 데이터셋 (features)

In [154]:
import pandas as pd
import numpy as np

# -----------------------------------
# 1. train 기준 전처리 규칙 학습
# -----------------------------------
def fit_preprocess_rules(train: pd.DataFrame) -> dict:
    train = train.copy()

    # 기본 name 파생
    train["family_name"] = train["name"].str.split(",").str[0].str.strip()
    train["title"] = train["name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()

    train["title"] = train["title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    rare_title_map = {
        "Lady": "Noble",
        "Countess": "Noble",
        "Sir": "Noble",
        "Dona": "Noble",
        "Don": "Noble",
        "Jonkheer": "Noble",
        "Capt": "Officer",
        "Col": "Officer",
        "Major": "Officer",
        "Dr": "Professional",
        "Rev": "Professional"
    }

    train["title_group"] = train["title"].replace(rare_title_map)

    # age 규칙
    family_age_median = train.groupby("family_name")["age"].median().to_dict()
    family_count = train.groupby("family_name")["family_name"].count().to_dict()
    pg_age_median = train.groupby(["pclass", "gender"])["age"].median().to_dict()
    global_age_median = train["age"].median()

    # fare 규칙
    fare_median = train["fare"].median()
    train_fare_filled = train["fare"].fillna(fare_median)

    _, fare_bins = pd.qcut(
        train_fare_filled,
        q=4,
        labels=["low", "mid", "high", "very_high"],
        retbins=True,
        duplicates="drop"
    )

    fare_bins = np.array(fare_bins, dtype=float)
    fare_bins[0] = -np.inf
    fare_bins[-1] = np.inf

    rules = {
        "rare_title_map": rare_title_map,
        "family_age_median": family_age_median,
        "family_count": family_count,
        "pg_age_median": pg_age_median,
        "global_age_median": global_age_median,
        "fare_median": fare_median,
        "fare_bins": fare_bins,
        "fare_labels": ["low", "mid", "high", "very_high"]
    }

    return rules


# -----------------------------------
# 2. age 채우기 함수
# -----------------------------------
def fill_age(row, rules):
    if pd.notnull(row["age"]):
        return row["age"]

    fam = row["family_name"]

    # 1) family_name 기준
    if rules["family_count"].get(fam, 0) >= 2:
        fam_med = rules["family_age_median"].get(fam, np.nan)
        if pd.notnull(fam_med):
            return fam_med

    # 2) pclass + gender 기준
    key = (row["pclass"], row["gender"])
    pg_med = rules["pg_age_median"].get(key, np.nan)
    if pd.notnull(pg_med):
        return pg_med

    # 3) 전체 median
    return rules["global_age_median"]


# -----------------------------------
# 3. feature 생성
# -----------------------------------
def add_hypothesis_features(df: pd.DataFrame, rules: dict) -> pd.DataFrame:
    df = df.copy()

    rare_title_map = rules["rare_title_map"]
    fare_median = rules["fare_median"]
    fare_bins = rules["fare_bins"]
    fare_labels = rules["fare_labels"]

    # 1. 기본 name 파생
    df["family_name"] = df["name"].str.split(",").str[0].str.strip()
    df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.", expand=False).str.strip()

    df["title"] = df["title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    df["title_group"] = df["title"].replace(rare_title_map)

    # 2. 가족 관련
    df["family_size"] = df["sibsp"] + df["parch"] + 1
    df["is_alone"] = (df["family_size"] == 1).astype(int)

    def family_size_group(x):
        if x == 1:
            return "alone"
        elif 2 <= x <= 4:
            return "small"
        else:
            return "large"

    df["family_size_group"] = df["family_size"].apply(family_size_group)

    # 3. age 결측치 처리
    df["age"] = df.apply(lambda r: fill_age(r, rules), axis=1)

    # 4. 성별 관련
    df["is_female"] = (df["gender"] == "female").astype(int)
    df["is_male"] = (df["gender"] == "male").astype(int)

    # 5. 선상 접근성 / 위치 근사

    df["deck"] = df["cabin"].fillna("Unknown").astype(str).apply(
    lambda x: x[0] if x != "Unknown" else "Unknown")
    df["deck"] = df["cabin"].str[0]


    df["has_cabin"] = df["cabin"].notnull().astype(int)

    df["upper_access"] = 0
    df.loc[df["pclass"] == 1, "upper_access"] = 2
    df.loc[df["pclass"] == 2, "upper_access"] = 1
    df.loc[df["pclass"] == 3, "upper_access"] = 0

    df["access_score"] = df["upper_access"] + df["has_cabin"]

    # 6. age / fare 구간화
    df["age_bin"] = pd.cut(
        df["age"],
        bins=[0,5,10,15,20,25, 30,35,40,45, 50,55,60,65,70,75,80,85,90],
        labels=False,
        include_lowest=True
    )

    df["fare"] = df["fare"].fillna(fare_median)

    df["fare_bin"] = pd.cut(
        df["fare"],
        bins=fare_bins,
        labels=fare_labels,
        include_lowest=True
    )

    def family_survival_zone(size):
        if size == 1:
            return "alone"
        elif 2 <= size <= 4:
            return "small"
        elif 5 <= size <= 7:
            return "medium"
        else:
            return "large"


    # 7. 조합 feature 0311
    df["is_child"] = (df["age"] < 16).astype(int)
    df["male_child"] = ((df["gender"] == "male") & (df["age"] < 16)).astype(int)
    df["female_child"] = ((df["gender"] == "female") & (df["age"] < 16)).astype(int)
    df["pclass_gender_has_cabin"] = ((df["is_female"] == 1) & (df["has_cabin"] == 0) & (df["pclass"] == 3)).astype(int) #0311
    df["child_with_family"] = ((df["age"] < 16) & (df["family_size"] > 1)).astype(int)
    df["fare_per_person"] = (df["fare"] / np.maximum(df["family_size"], 1)).astype(float)

    #######################################################################
    df["pclass_gender"] = df["pclass"].astype(str) + "_" + df["gender"].astype(str)            #0313
    df["pclass_title"] = df["pclass"].astype(str) + "_" + df["title_group"].astype(str)
    df["pclass_age_bin"] = df["pclass"].astype(str) + "_" + df["age_bin"].astype(str)
    df["pclass_is_male"] = df["pclass"].astype(str) + "_" + df["is_male"].astype(str)
    df["pclass_is_alone"] = df["pclass"].astype(str) + "_" + df["is_alone"].astype(str)
    df["access_pclass"] = df["access_score"].astype(str) + "_" + df["pclass"].astype(str)
    df["gender_title_group"] = df["gender"].astype(str) + "_" + df["title_group"].astype(str)        #0313
    df["gender_age_bin"] = df["gender"].astype(str) + "_" + df["age_bin"].astype(str)                #0313
    df["has_cabin_gender"] = df["has_cabin"].astype(str) + "_" + df["gender"].astype(str)                       #0311
    #df["ticket_farebin"] = df["ticket"].astype(str) + "_" + df["fare_bin"].astype(str)                          #0311
    df["family_survival_zone"] = df["family_size"].apply(family_survival_zone)
    df["female_child_pclass"] = df["female_child"].astype(str) + "_" + df["pclass"].astype(str)
    df["female_age_bin"] = df["is_female"].astype(str) + "_" + df["age_bin"].astype(str)
    df["female_familysize"] = df["is_female"].astype(str) + "_" + df["family_size"].astype(str)                #실험1
    df["male_familysize"] = df["is_male"].astype(str) + "_" + df["family_size"].astype(str)                    #실험1
    df["pclass_farebin"] = df["pclass"].astype(str) + "_" + df["fare_bin"].astype(str)
    #df["ticket_farebin_gender"] = df["gender"].astype(str) + "_" + df["ticket_farebin"].astype(str)
    df["gender_farebin"] =  df["gender"].astype(str) + "_" + df["fare_bin"].astype(str)
    #df["cabin_fare_bin"] = df["cabin"].astype(str) + "_" + df["fare_bin"].astype(str)                          #실험1
    #df["cabin_fare_bin_family_size"] = df["cabin_fare_bin"].astype(str) + "_" + df["family_size"].astype(str)
    #df["fare_bin_familysize"] = df["fare_bin"].astype(str) + "_" + df["family_size"].astype(str)
    df["pclass_gender_has_cabin"] = df["pclass_gender"].astype(str) + "_" + df["has_cabin"].astype(str)        #0311

    ####################################################################

    #df["surname"] = df["name"].str.split(",").str[0]
    #surname_survival = df.groupby("surname")["survived"].mean()
    #df["surname_survival_rate"] = df["surname"].map(surname_survival)


    ticket_counts = df.groupby("ticket")["ticket"].transform("count")

    df["ticket_group_size"] = ticket_counts

    df["ticket_group_size_bin"] = pd.cut(
        df["ticket_group_size"],
        bins=[0,1,2,4,10],
        labels=["solo","pair","small","large"]
    )

    ticket_count = df["ticket"].value_counts()

    df["ticket_group"] = df["ticket"].map(ticket_count)
    df["ticket_group"] = df["ticket_group"].apply(lambda x: 1 if x == 1 else 0)

    df["ticket_group_farebin"] = df["ticket_group"].astype(str) + "_" + df["fare_bin"].astype(str)


    return df

# EDA 분석 피쳐보기

In [155]:
# 1. train 기준 규칙 생성
rules = fit_preprocess_rules(train)

# 2. train / test / ori_te 모두 같은 규칙 적용
train = add_hypothesis_features(train, rules)
test = add_hypothesis_features(test, rules)
ori_te = add_hypothesis_features(ori_te, rules)

# 3. 확인
train.isnull().sum().sum(), test.isnull().sum().sum(), ori_te.isnull().sum().sum(),

(np.int64(1006), np.int64(431), np.int64(593))

In [156]:
train.shape

(641, 50)

In [157]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 641 entries, 539 to 636
Data columns (total 50 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   pclass                   641 non-null    int64   
 1   name                     641 non-null    object  
 2   gender                   641 non-null    object  
 3   age                      641 non-null    float64 
 4   sibsp                    641 non-null    int64   
 5   parch                    641 non-null    int64   
 6   ticket                   641 non-null    object  
 7   fare                     641 non-null    float64 
 8   cabin                    138 non-null    object  
 9   embarked                 641 non-null    object  
 10  family_name              641 non-null    object  
 11  title                    641 non-null    object  
 12  title_group              641 non-null    object  
 13  family_size              641 non-null    int64   
 14  is_alone     

In [158]:
drop_cols = ["name", "ticket", "cabin", "family_name", "is_male"]
#  is_male , upper_access, access_pclass, child_with_family, family_survival_zone
train = train.drop(columns=[c for c in drop_cols if c in train.columns])
test = test.drop(columns=[c for c in drop_cols if c in test.columns])
ori_te = ori_te.drop(columns=[c for c in drop_cols if c in ori_te.columns])

# 데이터프레임 정리

In [159]:
def prepare_frame(df: pd.DataFrame, numeric_cols: list, categorical_cols: list):
    df = df.copy()

    if "deck" in df.columns:
        df["deck"] = df["deck"].astype("object").fillna("Unknown")

    for col in categorical_cols:
        if col in df.columns:
            df[col] = df[col].astype("object").fillna("Unknown").astype(str)

    for col in numeric_cols:
        if col in df.columns and df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())

    return df

# 데이터 정리 함수

In [160]:
def get_feature_groups(df: pd.DataFrame):
    feature_cols = [
        "pclass",
        "age",
        "fare",
        "embarked",
        "title",
        "title_group",
        "family_size",
        "is_female",
        "deck",
        "upper_access",
        "access_score",
        "age_bin",
        "fare_per_person",
        "family_survival_zone",
        "ticket_group_size"
    ]

    cat_cols = [
        "embarked",
        "title",
        "title_group",
        "deck",
        "family_survival_zone"
    ]

    num_cols = [c for c in feature_cols if c not in cat_cols]

    return feature_cols, num_cols, cat_cols

# 전처리기 생성 함수

In [161]:
def encode_for_xgb(train_df, valid_df, test_df, cat_cols):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    for col in cat_cols:
        train_vals = train_df[col].astype(str).fillna("missing")
        valid_vals = valid_df[col].astype(str).fillna("missing")
        test_vals = test_df[col].astype(str).fillna("missing")

        uniq = sorted(train_vals.unique().tolist())
        mapping = {v: i for i, v in enumerate(uniq)}

        train_df[col] = train_vals.map(mapping).fillna(-1).astype(int)
        valid_df[col] = valid_vals.map(mapping).fillna(-1).astype(int)
        test_df[col] = test_vals.map(mapping).fillna(-1).astype(int)

    return train_df, valid_df, test_df

In [162]:
def encode_for_lgb(train_df, valid_df, test_df, cat_cols):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    for col in cat_cols:
        train_df[col] = train_df[col].astype("category")
        valid_df[col] = pd.Categorical(valid_df[col], categories=train_df[col].cat.categories)
        test_df[col] = pd.Categorical(test_df[col], categories=train_df[col].cat.categories)

    return train_df, valid_df, test_df

In [163]:
def encode_for_cat(train_df, valid_df, test_df, cat_cols):
    train_df = train_df.copy()
    valid_df = valid_df.copy()
    test_df = test_df.copy()

    for col in cat_cols:
        train_df[col] = train_df[col].astype(str).fillna("missing")
        valid_df[col] = valid_df[col].astype(str).fillna("missing")
        test_df[col] = test_df[col].astype(str).fillna("missing")

    return train_df, valid_df, test_df

# Training

In [164]:
reset_seeds()

# 모델 생성 함수

In [165]:
def get_base_models():
    xgb_model = XGBClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        min_child_weight=3,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=0.0,
        reg_alpha=0.3,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    )

    lgb_model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42
    )

    cat_model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        verbose=0,
        random_state=42
    )

    return xgb_model, lgb_model, cat_model


def get_meta_model():
    return LogisticRegression(
        max_iter=3000,
        random_state=42
    )

# threshold 탐색 함수

In [166]:
def find_best_threshold(y_true, pred_proba):
    best_th = 0.5
    best_f1 = -1

    for th in np.arange(0.35, 0.66, 0.01):
        pred = (pred_proba >= th).astype(int)
        score = f1_score(y_true, pred)

        if score > best_f1:
            best_f1 = score
            best_th = th

    return best_th, best_f1

# 평가 지표 함수

In [167]:
def evaluate_predictions(y_true, probas, threshold=0.5):
    pred = (probas >= threshold).astype(int)

    return {
        "threshold": threshold,
        "f1": f1_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "accuracy": accuracy_score(y_true, pred),
        "auc": roc_auc_score(y_true, probas)
    }

# Base model OOF 생성 함수

In [168]:
def generate_oof_predictions(X, y, X_test, cat_cols, n_splits=5, random_state=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_xgb = np.zeros(len(X))
    oof_lgb = np.zeros(len(X))
    oof_cat = np.zeros(len(X))

    test_xgb = np.zeros(len(X_test))
    test_lgb = np.zeros(len(X_test))
    test_cat = np.zeros(len(X_test))

    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        X_tr = X.iloc[tr_idx].copy()
        X_va = X.iloc[va_idx].copy()
        y_tr_fold = y.iloc[tr_idx].copy()
        y_va_fold = y.iloc[va_idx].copy()

        xgb_model, lgb_model, cat_model = get_base_models()

        # XGB
        X_tr_xgb, X_va_xgb, X_te_xgb = encode_for_xgb(X_tr, X_va, X_test, cat_cols)
        xgb_model.fit(X_tr_xgb, y_tr_fold)
        va_pred_xgb = xgb_model.predict_proba(X_va_xgb)[:, 1]
        te_pred_xgb = xgb_model.predict_proba(X_te_xgb)[:, 1]
        oof_xgb[va_idx] = va_pred_xgb
        test_xgb += te_pred_xgb / n_splits

        # LGB
        X_tr_lgb, X_va_lgb, X_te_lgb = encode_for_lgb(X_tr, X_va, X_test, cat_cols)
        lgb_model.fit(X_tr_lgb, y_tr_fold)
        va_pred_lgb = lgb_model.predict_proba(X_va_lgb)[:, 1]
        te_pred_lgb = lgb_model.predict_proba(X_te_lgb)[:, 1]
        oof_lgb[va_idx] = va_pred_lgb
        test_lgb += te_pred_lgb / n_splits

        # CAT
        X_tr_cat, X_va_cat, X_te_cat = encode_for_cat(X_tr, X_va, X_test, cat_cols)
        cat_features_idx = [X_tr_cat.columns.get_loc(c) for c in cat_cols]
        cat_model.fit(
            X_tr_cat,
            y_tr_fold,
            cat_features=cat_features_idx
        )
        va_pred_cat = cat_model.predict_proba(X_va_cat)[:, 1]
        te_pred_cat = cat_model.predict_proba(X_te_cat)[:, 1]
        oof_cat[va_idx] = va_pred_cat
        test_cat += te_pred_cat / n_splits

        fold_scores.append({
            "fold": fold,
            "xgb_auc": roc_auc_score(y_va_fold, va_pred_xgb),
            "lgb_auc": roc_auc_score(y_va_fold, va_pred_lgb),
            "cat_auc": roc_auc_score(y_va_fold, va_pred_cat),
        })

        print(
            f"[Fold {fold}] "
            f"XGB AUC={roc_auc_score(y_va_fold, va_pred_xgb):.5f} | "
            f"LGB AUC={roc_auc_score(y_va_fold, va_pred_lgb):.5f} | "
            f"CAT AUC={roc_auc_score(y_va_fold, va_pred_cat):.5f}"
        )

    oof_dict = {
        "xgb": oof_xgb,
        "lgb": oof_lgb,
        "cat": oof_cat
    }

    test_dict = {
        "xgb": test_xgb,
        "lgb": test_lgb,
        "cat": test_cat
    }

    fold_scores_df = pd.DataFrame(fold_scores)
    return oof_dict, test_dict, fold_scores_df

In [169]:
feature_cols, num_cols, cat_cols = get_feature_groups(train)

X = train[feature_cols].copy()
X_test = ori_te[feature_cols].copy()
y = y_tr.copy()

X.shape, X_test.shape, y.shape

((641, 15), (393, 15), (641,))

# OOF 예측 생성 실험

In [170]:
oof_dict, test_dict, fold_scores_df = generate_oof_predictions(
    X=X,
    y=y,
    X_test=X_test,
    cat_cols=cat_cols,
    n_splits=5,
    random_state=42
)

fold_scores_df

[LightGBM] [Info] Number of positive: 193, number of negative: 319
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000331 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 328
[LightGBM] [Info] Number of data points in the train set: 512, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.376953 -> initscore=-0.502501
[LightGBM] [Info] Start training from score -0.502501
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

,fold,xgb_auc,lgb_auc,cat_auc
0,1,0.906250,0.884821,0.894260
1,2,0.888802,0.884635,0.900000
2,3,0.864844,0.855729,0.865495
3,4,0.852995,0.853646,0.866146
4,5,0.919142,0.925859,0.899509


# base 모델 성능 확인

In [171]:
for model_name, preds in oof_dict.items():
    th, best_f1 = find_best_threshold(y, preds)
    auc = roc_auc_score(y, preds)
    acc = accuracy_score(y, (preds >= th).astype(int))

    print(
        f"{model_name.upper()} | "
        f"best_th={th:.2f} | "
        f"OOF F1={best_f1:.5f} | "
        f"OOF AUC={auc:.5f} | "
        f"ACC={acc:.5f}"
    )

XGB | best_th=0.53 | OOF F1=0.82198 | OOF AUC=0.88378 | ACC=0.87363
LGB | best_th=0.50 | OOF F1=0.81579 | OOF AUC=0.87852 | ACC=0.86895
CAT | best_th=0.60 | OOF F1=0.81938 | OOF AUC=0.88214 | ACC=0.87207


# meta feature 생성 셀

In [172]:
for model_name, preds in oof_dict.items():
    th, best_f1 = find_best_threshold(y, preds)
    auc = roc_auc_score(y, preds)
    acc = accuracy_score(y, (preds >= th).astype(int))

    print(
        f"{model_name.upper()} | "
        f"best_th={th:.2f} | "
        f"OOF F1={best_f1:.5f} | "
        f"OOF AUC={auc:.5f} | "
        f"ACC={acc:.5f}"
    )

XGB | best_th=0.53 | OOF F1=0.82198 | OOF AUC=0.88378 | ACC=0.87363
LGB | best_th=0.50 | OOF F1=0.81579 | OOF AUC=0.87852 | ACC=0.86895
CAT | best_th=0.60 | OOF F1=0.81938 | OOF AUC=0.88214 | ACC=0.87207


# stacking meta model셀

In [173]:
meta_model = get_meta_model()
meta_model.fit(X_meta, y)

meta_oof_pred = meta_model.predict_proba(X_meta)[:, 1]
meta_test_pred = meta_model.predict_proba(X_meta_test)[:, 1]

meta_best_th, meta_best_f1 = find_best_threshold(y, meta_oof_pred)
meta_auc = roc_auc_score(y, meta_oof_pred)
meta_acc = accuracy_score(y, (meta_oof_pred >= meta_best_th).astype(int))

print(f"STACK META | best_th={meta_best_th:.2f}")
print(f"STACK META | OOF F1={meta_best_f1:.5f}")
print(f"STACK META | OOF AUC={meta_auc:.5f}")
print(f"STACK META | OOF ACC={meta_acc:.5f}")

NameError: name 'X_meta' is not defined

# stacking submission 저장 셀

In [ ]:
final_test_pred = (meta_test_pred >= meta_best_th).astype(int)

submission_df = pd.DataFrame({
    "passengerid": ori_te.index,
    "survived": final_test_pred
})

assert submission_df["survived"].isnull().sum() == 0, "NaN 존재"

stack_submission_path = "results_stacking_submission.csv"
submission_df.to_csv(stack_submission_path, index=False)

print("저장 완료:", stack_submission_path)
submission_df.head()

# soft blend 셀

In [ ]:
blend_oof_pred = (
    0.45 * oof_dict["xgb"] +
    0.25 * oof_dict["lgb"] +
    0.30 * oof_dict["cat"]
)

blend_test_pred = (
    0.45 * test_dict["xgb"] +
    0.25 * test_dict["lgb"] +
    0.30 * test_dict["cat"]
)

blend_best_th, blend_best_f1 = find_best_threshold(y, blend_oof_pred)
blend_auc = roc_auc_score(y, blend_oof_pred)
blend_acc = accuracy_score(y, (blend_oof_pred >= blend_best_th).astype(int))

print(f"BLEND | best_th={blend_best_th:.2f}")
print(f"BLEND | OOF F1={blend_best_f1:.5f}")
print(f"BLEND | OOF AUC={blend_auc:.5f}")
print(f"BLEND | OOF ACC={blend_acc:.5f}")

# blend submission 저장 셀

In [ ]:
blend_final_test_pred = (blend_test_pred >= blend_best_th).astype(int)

submission_blend = pd.DataFrame({
    "passengerid": ori_te.index,
    "survived": blend_final_test_pred
})

blend_submission_path = "results_blend_submission.csv"
submission_blend.to_csv(blend_submission_path, index=False)

print("저장 완료:", blend_submission_path)
submission_blend.head()